<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/experts_upstairts_fashion_seuil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TÉLÉCHARGEMENT FASHION-MNIST (une seule fois par session Colab)
# ══════════════════════════════════════════════════════════════════════════════
import os
os.makedirs("fmnist", exist_ok=True)
!wget -q -O fmnist/t10k-images-idx3-ubyte.gz https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/t10k-images-idx3-ubyte.gz
!wget -q -O fmnist/t10k-labels-idx1-ubyte.gz https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/t10k-labels-idx1-ubyte.gz
!wget -q -O fmnist/train-images-idx3-ubyte.gz https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/train-images-idx3-ubyte.gz
!wget -q -O fmnist/train-labels-idx1-ubyte.gz https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/train-labels-idx1-ubyte.gz
print("Fashion-MNIST téléchargé dans fmnist/")


Fashion-MNIST téléchargé dans fmnist/


In [ ]:
import numpy as np
import time
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.optimize import minimize, LinearConstraint

# ══════════════════════════════════════════════════════════════════════════════
# PLONGEMENT SPECTRAL vs DEUX NAPPES  {P_sep=0} ∪ {P_sep=ε}
#
# Question : les deux nappes S'ÉLOIGNENT-elles dans le plongement ?
# Protocole :
#   1. cloud 50/50 sur les deux nappes (P_sep renormalisé std=1, Gauss-Newton) ;
#   2. diagnostic de séparation AVANT (espace ambiant) ;
#   3. classification QP (semi-interpolation Sobolev, expert polynomial) AVANT ;
#   4. plongement spectral (projections aléatoires + base polynomiale + Gram
#      Sobolev, vecteurs propres normalisés) — le programme historique ;
#   5. diagnostic de séparation APRÈS (espace plongé) ;
#   6. classification QP APRÈS (petit degré : si le plongement redresse, un
#      degré bas doit suffire) ;
#   7. baseline Ridge poly avant/après (RBF supprimé).
# ══════════════════════════════════════════════════════════════════════════════

params = {
    # ── DONNÉES (Fashion-MNIST 28×28 -> img_size×img_size) ────────────────────
    "fmnist_neg":  [4],         # t-shirt=0 (classe -1) veste=4
    "fmnist_pos":  [2],         # pullover (classe +1)
    "img_size":    7,           # sous-échantillonnage 28×28 -> img_size×img_size
                                #   7 -> dim 49 (par moyenne de blocs 4×4)
    "n_train":     10,          # points étiquetés (n/2 par classe)
    "n_test":      200,
    "n_unlabeled": 2000,        # non-étiquetés (informe la norme H du cloud)
    "seed":        434772,

    # ── NORME DE SOBOLEV (référence QP ambiant) ───────────────────────────────
    "weights":      {0: 1e-4, 1: 1, 2: 0, 3: 0},

    # ── CLASSIFICATION QP (référence ambiant + réutilisée pour experts) ───────
    "qp_margin":     1.0,
    "deg_clf_avant": 2,
    "thres1":        1e-15,
    "thres2":        1e3,
    "const_pen":     0,
    "n_G":           1500,

    # ── EXPERTS PAR PROJECTIONS ALÉATOIRES ────────────────────────────────────
    "n_experts":    49,          # nombre de projections/experts
    "k_proj":       1,          # dim de chaque projection
    "n_deg":        8,          # degré des polynômes par expert
    "lambda_G":     1.0,        # régularisation Gram
    "weights_plg":  {0: 0, 1: 1, 2: 0, 3: 0},   # poids Sobolev (w1 seul)
    # Plongement spectral par expert : chaque g_i est cherché dans le sous-espace
    # engendré par les PLUS PETITES valeurs propres de la Gram Sobolev projetée
    # (les directions les plus lisses sur π(cloud)).
    "emb_thres1_expert": 0.9,   # plancher ABSOLU pour les VP (évite le noyau numérique)
    "emb_thres2_expert": 50,
    "embed_dim_expert":  2000,    # cap sur le nombre de modes retenus par expert
    # Phase 2 : sélecteur d'experts par ‖g‖_H
    "K_norm_selector":  1e3,    # jette l'expert si ‖g‖_H > K · min(‖g‖_H)
                                #   mettre np.inf pour désactiver le sélecteur
}

# ══════════════════════════════════════════════════════════════════════════════
# CHARGEMENT DES DONNÉES : digits sklearn, chiffres a vs b, réduits en 4×4
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.datasets import load_digits

def _load_fmnist():
    """Charge Fashion-MNIST depuis les fichiers idx locaux (dossier ./fmnist/).
    Retourne (X, y) avec X shape (N, 28, 28), y (N,) dtype int."""
    import gzip, os
    paths = ["fmnist", "./fmnist"]
    base = next((p for p in paths if os.path.isdir(p)), None)
    if base is None:
        raise FileNotFoundError(
            "Dossier fmnist/ introuvable. Téléchargez les 4 fichiers idx depuis "
            "https://github.com/zalandoresearch/fashion-mnist/tree/master/data/fashion")
    def _read_img(p):
        with gzip.open(p, 'rb') as f:
            _ = f.read(16); return np.frombuffer(f.read(), dtype=np.uint8).reshape(-1, 28, 28)
    def _read_lab(p):
        with gzip.open(p, 'rb') as f:
            _ = f.read(8); return np.frombuffer(f.read(), dtype=np.uint8)
    X = np.vstack([_read_img(f"{base}/t10k-images-idx3-ubyte.gz"),
                   _read_img(f"{base}/train-images-idx3-ubyte.gz")])
    y = np.concatenate([_read_lab(f"{base}/t10k-labels-idx1-ubyte.gz"),
                        _read_lab(f"{base}/train-labels-idx1-ubyte.gz")])
    return X.astype(float), y.astype(int)

def load_fmnist_binary(neg_list, pos_list, img_size,
                       n_train, n_test, n_unlabeled, seed=42):
    """Charge Fashion-MNIST, garde neg_list vs pos_list, sous-échantillonne
    28×28 -> img_size×img_size par moyenne de blocs (28/img_size)². Normalise
    chaque feature. Split équilibré train/test/unlabeled."""
    X28, y_raw = _load_fmnist()
    keep = set(neg_list) | set(pos_list)
    mask = np.isin(y_raw, list(keep))
    X28 = X28[mask]; y_raw = y_raw[mask]
    y = np.where(np.isin(y_raw, list(pos_list)), 1.0, -1.0)
    # sous-échantillonnage : 28 doit être divisible par img_size
    if 28 % img_size != 0:
        raise ValueError(f"img_size={img_size} ne divise pas 28")
    b = 28 // img_size
    Xs = X28.reshape(-1, img_size, b, img_size, b).mean(axis=(2, 4))
    Xs = Xs.reshape(-1, img_size*img_size)
    # normalisation par feature (centrage + std)
    Xs = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-8)
    rng = np.random.default_rng(seed)
    idx_neg = np.where(y < 0)[0]; idx_pos = np.where(y > 0)[0]
    rng.shuffle(idx_neg); rng.shuffle(idx_pos)
    n_tr_c = n_train // 2; n_te_c = n_test // 2
    tr = np.concatenate([idx_neg[:n_tr_c], idx_pos[:n_tr_c]])
    te = np.concatenate([idx_neg[n_tr_c:n_tr_c+n_te_c],
                         idx_pos[n_tr_c:n_tr_c+n_te_c]])
    used = set(tr) | set(te)
    unlab_pool = np.array([i for i in range(len(Xs)) if i not in used])
    if len(unlab_pool) > n_unlabeled:
        unlab = rng.choice(unlab_pool, n_unlabeled, replace=False)
    else:
        unlab = unlab_pool
    rng.shuffle(tr); rng.shuffle(te); rng.shuffle(unlab)
    return Xs[tr], y[tr], Xs[te], y[te], Xs[unlab], y[unlab]

_labels = {0:"t-shirt", 1:"pantalon", 2:"pullover", 3:"robe", 4:"manteau",
           5:"sandale", 6:"chemise", 7:"sneaker", 8:"sac", 9:"bottine"}
_neg_s = "{" + ",".join(_labels[c] for c in params["fmnist_neg"]) + "}"
_pos_s = "{" + ",".join(_labels[c] for c in params["fmnist_pos"]) + "}"

X_train, y_train, X_test, y_test, X_unlab, y_unlab = load_fmnist_binary(
    params["fmnist_neg"], params["fmnist_pos"], params["img_size"],
    params["n_train"], params["n_test"], params["n_unlabeled"],
    seed=params["seed"])
d = X_train.shape[1]
X_all = np.vstack([X_train, X_unlab]); N_all = len(X_all)
y_all = np.concatenate([y_train, y_unlab])
print(f"Fashion-MNIST {_neg_s} vs {_pos_s}, "
      f"{params['img_size']}×{params['img_size']} -> dim {d}")
print(f"  train    : {len(X_train)} pts  ({int((y_train<0).sum())} × {_neg_s}, "
      f"{int((y_train>0).sum())} × {_pos_s})")
print(f"  test     : {len(X_test)} pts  ({int((y_test<0).sum())} × {_neg_s}, "
      f"{int((y_test>0).sum())} × {_pos_s})")
print(f"  unlabeled: {len(X_unlab)} pts  (labels cachés pour le fit)")

# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC DE SÉPARATION  (avant / après plongement)
#   ratio inter/intra : distance moyenne entre nappes / distance moyenne au sein
#   d'une nappe. > 1 = les nappes s'éloignent. On mesure sur le cloud (labels de
#   génération, diagnostic pur).
# ══════════════════════════════════════════════════════════════════════════════
def separation_diag(Z, y, n_sub=800, seed=0, titre=""):
    rng = np.random.default_rng(seed)
    i0 = np.where(y < 0)[0]; i1 = np.where(y > 0)[0]
    i0 = rng.choice(i0, min(n_sub, len(i0)), replace=False)
    i1 = rng.choice(i1, min(n_sub, len(i1)), replace=False)
    Z0, Z1 = Z[i0], Z[i1]
    def mdist(A, B):
        return np.mean(np.sqrt(np.maximum(
            np.sum(A**2,1)[:,None] + np.sum(B**2,1)[None,:] - 2*A@B.T, 0)))
    intra = 0.5*(mdist(Z0, Z0) + mdist(Z1, Z1))
    inter = mdist(Z0, Z1)
    # 1-NN inter-nappe : fraction de points dont le plus proche voisin est de
    # l'AUTRE nappe (0 = nappes bien séparées, ~0.5 = mélangées)
    Zs = np.vstack([Z0, Z1]); ys = np.concatenate([-np.ones(len(Z0)), np.ones(len(Z1))])
    D = np.sum(Zs**2,1)[:,None] + np.sum(Zs**2,1)[None,:] - 2*Zs@Zs.T
    np.fill_diagonal(D, np.inf)
    nn = np.argmin(D, axis=1)
    frac_cross = np.mean(ys[nn] != ys)
    print(f"  [{titre}] inter/intra = {inter/intra:.4f}   "
          f"(inter={inter:.3f} intra={intra:.3f})   1-NN croisé = {frac_cross:.3f}")
    return inter/intra, frac_cross

# ══════════════════════════════════════════════════════════════════════════════
# CLASSIFICATEUR QP  (semi-interpolation Sobolev, expert polynomial générique)
#   min ‖u‖²_H  s.c.  y_i·u(x_i) >= marge,  base H-orthonormée + constante libre.
#   Générique : X de dimension quelconque (ambiant OU plongé).
# ══════════════════════════════════════════════════════════════════════════════
def solve_qp(A, y, G, margin, verbose=False):
    n, k = A.shape
    Gr = G + 1e-12*np.eye(k)
    con = LinearConstraint(np.diag(y)@A, lb=margin, ub=np.inf)
    try:    c0 = np.linalg.lstsq(A, 1.5*margin*y, rcond=None)[0]
    except Exception: c0 = np.zeros(k)
    res = minimize(lambda c: c@Gr@c, c0, jac=lambda c: 2*Gr@c,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-11})
    c = res.x
    marge_eff = float(np.min(y*(A@c)))
    return c, marge_eff >= margin - 1e-4, marge_eff

def classify_poly_qp(X_tr, y_tr, X_te, y_te, X_cloud, deg, weights, titre=""):
    """Classifieur polynomial par semi-interpolation QP. Retourne (acc, auc)."""
    n, dd = X_tr.shape
    # normalisation par coordonnée -> [-1,1] (calculée sur le cloud)
    lo = X_cloud.min(axis=0); hi = X_cloud.max(axis=0)
    span = np.where(hi - lo > 1e-12, hi - lo, 1.0)
    def to_u(X): return 2*(X - lo)/span - 1.0
    SC = 2.0/span                                      # ∂u/∂x par coordonnée

    poly = PolynomialFeatures(degree=deg, include_bias=False)
    Phi_tr = poly.fit_transform(to_u(X_tr))
    powers = poly.powers_; n_feat = Phi_tr.shape[1]

    def pderiv(U, coefs, dims=()):
        P = powers.astype(float).copy(); m = coefs.astype(float).copy()
        for dm in dims: m = m*P[:, dm]; P[:, dm] -= 1
        v = m != 0
        return np.zeros(len(U)) if not v.any() else (U[:, None, :]**P[None, v, :]).prod(2)@m[v]

    rng = np.random.default_rng(4242)
    idx = rng.choice(len(X_cloud), min(params["n_G"], len(X_cloud)), replace=False)
    U_G = to_u(X_cloud[idx]); nG = len(idx)
    PHI = poly.transform(U_G)
    E = np.eye(n_feat)
    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.); w2 = weights.get(2, 0.)
    G = np.zeros((n_feat, n_feat))
    if w0: G += w0*(PHI.T@PHI)/nG
    if w1:
        GG = np.zeros((nG, n_feat, dd))
        for j in range(n_feat):
            for a in range(dd):
                if powers[j, a] > 0:
                    GG[:, j, a] = SC[a]*pderiv(U_G, E[j], (a,))
        G += w1*np.einsum('xik,xjk->ij', GG, GG)/nG
    if w2:
        HH = np.zeros((nG, n_feat, dd, dd))
        for j in range(n_feat):
            for a in range(dd):
                for b in range(a, dd):
                    if (powers[j, a] > 0 and powers[j, b] > 0) or (a == b and powers[j, a] > 1):
                        v = SC[a]*SC[b]*pderiv(U_G, E[j], (a, b))
                        HH[:, j, a, b] = v; HH[:, j, b, a] = v
        G += w2*np.einsum('xikl,xjkl->ij', HH, HH)/nG

    # base H-orthonormée (bande) + centrage + constante libre ancrée
    mean_phi = PHI.mean(axis=0)
    s_g, V_g = np.linalg.eigh(G); smax = 1 #s_g.max()
    keep = (s_g > smax*params["thres1"]) & (s_g < smax*params["thres2"])
    if keep.sum() == 0:
        raise ValueError("bande spectrale vide")
    T = V_g[:, keep]/np.sqrt(s_g[keep]); r = int(keep.sum())
    A_qp = np.hstack([(Phi_tr - mean_phi)@T, np.ones((n, 1))])
    G_qp = np.eye(r+1); G_qp[-1, -1] = params["const_pen"]
    sol, feas, marge = solve_qp(A_qp, y_tr, G_qp, params["qp_margin"])
    coef = T@sol[:r]; off = sol[-1] - mean_phi@coef

    f_te = poly.transform(to_u(X_te))@coef + off
    f_tr = Phi_tr@coef + off
    def acc(f, y):
        sg = np.sign(f)
        return np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float)))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f_te)
    except Exception: auc = float('nan')
    print(f"  [{titre}] deg={deg} feat={n_feat} rang={r} | faisable={feas} "
          f"marge={marge:.3f} ‖u‖_H={np.sqrt(max(sol[:r]@sol[:r],0)):.4f} | "
          f"acc_tr={acc(f_tr,y_tr):.2f} acc_te={acc(f_te,y_te):.4f} AUC={auc:.4f}")
    return acc(f_te, y_te), auc

def classify_ridge(X_tr, y_tr, X_te, y_te, deg, titre=""):
    poly = PolynomialFeatures(degree=deg)
    r = Ridge(alpha=1e-8).fit(poly.fit_transform(X_tr), y_tr)
    f = r.predict(poly.transform(X_te))
    a = np.mean(np.sign(f) == np.sign(y_te))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f)
    except Exception: auc = float('nan')
    print(f"  [{titre}] Ridge deg={deg} : acc_te={a:.4f} AUC={auc:.4f}")
    return a, auc

# ══════════════════════════════════════════════════════════════════════════════
# PLONGEMENT SPECTRAL  (programme historique, repris tel quel)
# ══════════════════════════════════════════════════════════════════════════════
def activate(X, mode):
    if mode == "tanh":    return np.tanh(X)
    if mode == "sigmoid": return 1/(1 + np.exp(-X))
    return X

def make_projections_ortho(dd, k_proj, rng):
    """Projections sur des sous-espaces ORTHOGONAUX deux à deux qui PARTITIONNENT
    R^dd. Construction : QR d'une matrice gaussienne dd×dd -> base orthonormée
    aléatoire ; on la découpe en blocs de k_proj lignes. Le dernier bloc a
    dimension dd mod k_proj s'il ne divise pas (choix unique : l'orthogonal des
    précédents). k_proj est PLAFONNÉ à dd (au-delà, les vecteurs propres seraient
    redondants pour beaucoup de calcul en plus).
    GARANTIE : la concaténation des bases relevées contient TOUS les polynômes de
    degré 1 dans son span — les lignes des blocs forment une base de R^dd, donc
    toute forme linéaire x ↦ v·x est combinaison des coordonnées projetées."""
    k = min(k_proj, dd)
    M = rng.standard_normal((dd, dd))
    Q, _ = np.linalg.qr(M)                     # lignes de Q.T = base orthonormée
    B = Q.T                                    # (dd, dd) : chaque ligne unitaire
    blocks = [B[i:i+k] for i in range(0, dd, k)]
    return blocks                              # liste de (k_p, dd), k_p<=k

def poly_features_projected(X, Pis, n_deg):
    """Pis : liste de matrices (k_p, d). Monômes de degré <= n_deg par projection
    (k_p peut varier -> monômes par bloc). Retourne Phi concaténée + la liste des
    monômes par bloc."""
    n = X.shape[0]
    cols = []; monomes_list = []
    for Pi_p in Pis:
        k_p = Pi_p.shape[0]
        Zp = X @ Pi_p.T                        # (n, k_p)
        monomes = [a for a in product(range(n_deg+1), repeat=k_p) if 0 < sum(a) <= n_deg]
        monomes_list.append(monomes)
        Phi_p = np.ones((n, len(monomes)))
        for m, alpha in enumerate(monomes):
            for j, e in enumerate(alpha):
                if e > 0: Phi_p[:, m] *= Zp[:, j]**e
        cols.append(Phi_p)
    return np.hstack(cols), monomes_list

def poly_gram_projected(X_cloud, Pis, n_deg, weights, lambda_G):
    n = X_cloud.shape[0]
    sizes = []
    monomes_list = []
    for Pi_p in Pis:
        k_p = Pi_p.shape[0]
        monomes = [a for a in product(range(n_deg+1), repeat=k_p) if 0 < sum(a) <= n_deg]
        monomes_list.append(monomes); sizes.append(len(monomes))
    n_funcs = sum(sizes)
    G = np.zeros((n_funcs, n_funcs))
    off = 0
    for Pi_p, monomes in zip(Pis, monomes_list):
        k_p = Pi_p.shape[0]; n_mon = len(monomes)
        sl = slice(off, off + n_mon); off += n_mon
        Zp = X_cloud @ Pi_p.T
        Phi_p = np.ones((n, n_mon))
        for m, alpha in enumerate(monomes):
            for j, e in enumerate(alpha):
                if e > 0: Phi_p[:, m] *= Zp[:, j]**e
        w0 = weights.get(0, 0.)
        if w0: G[sl, sl] += w0*(Phi_p.T@Phi_p)/n
        w1 = weights.get(1, 0.)
        if w1:
            GradZ = np.zeros((n, n_mon, k_p))
            for m, alpha in enumerate(monomes):
                for j in range(k_p):
                    if alpha[j] > 0:
                        col = np.ones(n)*alpha[j]
                        for l, e in enumerate(alpha):
                            e2 = e - 1 if l == j else e
                            if e2 > 0: col *= Zp[:, l]**e2
                        GradZ[:, m, j] = col
            GradX = np.einsum('nmj,jd->nmd', GradZ, Pi_p)
            G[sl, sl] += w1*np.einsum('nmi,nli->ml', GradX, GradX)/n
    G += lambda_G*np.eye(n_funcs)
    return G

def poly_embedding(X_input, X_cloud, weights, lambda_G,
                   k_proj, n_deg, embed_dim, rng,
                   emb_thres1, emb_thres2):
    """Plongement spectral. Projections orthogonales partitionnant l'espace
    (n_proj dérivé = ceil(d/k_proj)). Sélection des vecteurs propres par BANDE :
    on garde emb_thres1 < s/s_max < emb_thres2 (plancher ET plafond — les plus
    grandes VP sont dominées par les fonctions de grande norme H, à jeter ;
    les plus petites sont le noyau numérique). Cap à embed_dim modes."""
    Pis = make_projections_ortho(X_input.shape[1], k_proj, rng)
    print(f"  projections orthogonales : {len(Pis)} blocs de dims "
          f"{[p.shape[0] for p in Pis]} (partition de R^{X_input.shape[1]})")
    Phi_cloud, monomes = poly_features_projected(X_cloud, Pis, n_deg)
    G = poly_gram_projected(X_cloud, Pis, n_deg, weights, lambda_G)
    eigvals, eigvecs = np.linalg.eigh(G)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]; eigvecs = eigvecs[:, order]
    smax = 1  # échelle absolue
    # bande [plancher, plafond] activée. On garde les embed_dim DERNIERS
    # (plus petites VP) de la bande — les directions les plus lisses sur le cloud.
    band_idx = np.where((eigvals > emb_thres1*smax) & (eigvals < emb_thres2*smax))[0]
    if len(band_idx) == 0:
        raise ValueError(f"bande vide : s ∈ [{eigvals.min():.2e}, {eigvals.max():.2e}] ; "
                         f"ajuster emb_thres1/emb_thres2")
    idx_sel = band_idx[-embed_dim:] if len(band_idx) > embed_dim else band_idx
    V = eigvecs[:, idx_sel]; L = eigvals[idx_sel]
    n_above = int((eigvals >= emb_thres2*smax).sum())
    n_below = int((eigvals <= emb_thres1*smax).sum())
    print(f"  spectre : {len(eigvals)} VP | "
          f"jetées : {n_above} en haut (s>={emb_thres2:g}), "
          f"{n_below} en bas (s<={emb_thres1:g}) | "
          f"disponibles dans la bande : {len(band_idx)} | retenues : {len(L)}")
    print(f"  VP gardées ({len(L)}) : {np.array2string(L, precision=3, max_line_width=100)}")
    V_norm = V/np.sqrt(L)[None, :]
    # centrer les coordonnées du plongement (moyenne nulle sur le cloud)
    embed_cld = Phi_cloud@V_norm
    embed_means = embed_cld.mean(axis=0)     # mémorisé pour les nouvelles données
    Phi_in, _ = poly_features_projected(X_input, Pis, n_deg)
    return Phi_in@V_norm - embed_means, Pis, V_norm, embed_means

# ══════════════════════════════════════════════════════════════════════════════
# EXPÉRIENCE
# ══════════════════════════════════════════════════════════════════════════════
# centrage (comme le programme historique) — le plongement travaille centré
center = X_all.mean(axis=0)
Xc_train = X_train - center; Xc_test = X_test - center; Xc_all = X_all - center

print("\n" + "="*72)
print("AVANT PLONGEMENT (espace ambiant)")
print("="*72)
sep0 = separation_diag(Xc_all, y_all, titre="séparation ambiant")
acc0, auc0 = classify_poly_qp(Xc_train, y_train, Xc_test, y_test, Xc_all,
                              params["deg_clf_avant"], params["weights"],
                              titre="QP ambiant")
classify_ridge(Xc_train, y_train, Xc_test, y_test, 3, titre="baseline ambiant")

print("\n" + "="*72)
print(f"EXPERTS PAR PROJECTIONS ALÉATOIRES  "
      f"(n_experts={params['n_experts']}, k_proj={params['k_proj']}, "
      f"n_deg={params['n_deg']})")
print("="*72)

# ── Chaque expert i ──────────────────────────────────────────────────────────
# π_i : R^N -> R^k projection orthogonale aléatoire (lignes orthonormées)
# g_i : polynôme sur R^k, obtenu par QP semi-interpolation sur le cloud projeté
# f_i(x) = g_i(π_i · x)  (fonction sur l'espace ambiant, par composition)
# Les dérivées transversales à π_i sont nulles -> la Gram Sobolev de f_i
# sur le cloud (produit scalaire H(cloud)) coïncide avec celle de g_i sur le
# cloud projeté (π_i orthogonale). C'est ce qui rend l'approche cohérente.
#
# Phase 2 : QP sur les f_i avec la Gram H(cloud) des experts,
#   G_H[i,j] = <f_i, f_j>_H(cloud)
#            = Σ_o w_o · mean_cloud(∂^o f_i · ∂^o f_j).
# Pour deux experts avec projections π_i, π_j différentes :
#   ∇_x f_i = π_i^T · ∇_z g_i
#   <∇f_i, ∇f_j>_x = ∇g_i^T (π_i π_j^T) ∇g_j  (produit croisé non trivial)
# On calcule tout par Monte Carlo direct sur X_cloud : Φ_i(X) et Grad_i(X) via
# la composition — pas besoin de manipuler π_i π_j^T explicitement.
# ──────────────────────────────────────────────────────────────────────────────

def random_projection(n_amb, k, rng):
    """Projection orthogonale aléatoire R^n -> R^k. Retourne P de shape (k, n)
    avec lignes orthonormées (P P^T = I_k)."""
    M = rng.standard_normal((n_amb, k))
    Q, _ = np.linalg.qr(M)   # colonnes de Q : orthonormées (n, k)
    return Q.T                # (k, n) : lignes orthonormées

def poly_features_and_grad(Z, monomes):
    """Features polynomiales + gradient sur Z (n, k). Retourne (Phi, Grad)
    avec Phi (n, nf) et Grad (n, nf, k). Version vectorisée."""
    n, k = Z.shape; nf = len(monomes)
    alphas = np.array(monomes, dtype=np.int32)          # (nf, k)
    # Phi_ij = prod_l Z[i,l]^alphas[j,l]
    #        = exp(sum_l alphas[j,l] * log(Z[i,l]))  — pas praticable si Z<=0
    # On calcule par multiplication vectorisée sur les dimensions.
    Phi = np.ones((n, nf))
    for l in range(k):
        e = alphas[:, l]                                 # (nf,)
        maxe = int(e.max())
        if maxe == 0: continue
        # tableau des puissances : powers[p] = Z[:, l]**p pour p=0..maxe
        powers = np.empty((maxe+1, n))
        powers[0] = 1.0; powers[1] = Z[:, l]
        for p in range(2, maxe+1):
            powers[p] = powers[p-1] * Z[:, l]
        Phi *= powers[e].T                               # (n, nf)
    # Gradient : ∂Phi_j / ∂Z_l = alpha[j,l] * Phi_j / Z[:, l]  (formellement)
    # Mais Z peut avoir des zéros -> on refait le produit sans la coord l.
    Grad = np.zeros((n, nf, k))
    for l in range(k):
        e = alphas[:, l]                                 # (nf,)
        if e.max() == 0: continue
        # phi_without_l : produit des autres dimensions
        phi_wo = np.ones((n, nf))
        for lp in range(k):
            if lp == l: continue
            ep = alphas[:, lp]; maxep = int(ep.max())
            if maxep == 0: continue
            powers = np.empty((maxep+1, n))
            powers[0] = 1.0; powers[1] = Z[:, lp]
            for p in range(2, maxep+1):
                powers[p] = powers[p-1] * Z[:, lp]
            phi_wo *= powers[ep].T
        # dérivée = alpha_l * Z_l^(alpha_l - 1) * phi_wo
        maxe = int(e.max())
        powers = np.empty((maxe+1, n))
        powers[0] = 1.0
        if maxe >= 1: powers[1] = Z[:, l]
        for p in range(2, maxe+1):
            powers[p] = powers[p-1] * Z[:, l]
        e_minus = np.maximum(e - 1, 0)                   # évite index -1
        Zpow = powers[e_minus].T                         # (n, nf)
        Grad[:, :, l] = e[None, :].astype(float) * Zpow * phi_wo
    return Phi, Grad


def fit_expert(X_cloud, X_train, y_train, P, n_deg, weights, lambda_G,
    qp_margin, thres1, thres2, const_pen,
    emb_thres1_expert, embed_dim_expert):
    """Expert = QP semi-interpolation dans l'espace projeté Z = X P^T, calqué
    exactement sur classify_poly_qp (PolynomialFeatures + pderiv vectorisés),
    donc valable pour n_deg quelconque sans boucle Python sur les monômes."""
    k = P.shape[0]
    Z_cloud = X_cloud @ P.T
    Z_train = X_train @ P.T
    lo = Z_cloud.min(axis=0); hi = Z_cloud.max(axis=0)
    span = np.where(hi - lo > 1e-12, hi - lo, 1.0)
    def to_u(Z): return 2*(Z - lo)/span - 1.0
    SC = 2.0/span                                     # ∂u/∂z par coordonnée

    poly = PolynomialFeatures(degree=n_deg, include_bias=False)
    Phi_train = poly.fit_transform(to_u(Z_train))
    powers = poly.powers_; n_feat = Phi_train.shape[1]

    def pderiv(U, coefs, dims=()):
        Pw = powers.astype(float).copy(); m = coefs.astype(float).copy()
        for dm in dims: m = m*Pw[:, dm]; Pw[:, dm] -= 1
        v = m != 0
        return np.zeros(len(U)) if not v.any() else (U[:, None, :]**Pw[None, v, :]).prod(2)@m[v]

    rng = np.random.default_rng(4242)
    idx = rng.choice(len(Z_cloud), min(params["n_G"], len(Z_cloud)), replace=False)
    U_G = to_u(Z_cloud[idx]); nG = len(idx)
    PHI = poly.transform(U_G)
    E = np.eye(n_feat)
    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    G = np.zeros((n_feat, n_feat))
    if w0: G += w0*(PHI.T@PHI)/nG
    if w1:
        GG = np.zeros((nG, n_feat, k))
        for j in range(n_feat):
            for a in range(k):
                if powers[j, a] > 0:
                    GG[:, j, a] = SC[a]*pderiv(U_G, E[j], (a,))
        G += w1*np.einsum('xik,xjk->ij', GG, GG)/nG
    G += lambda_G*np.eye(n_feat)

    mean_phi = PHI.mean(axis=0)
    s_g, V_g = np.linalg.eigh(G); smax = 1
    keep = (s_g > smax*thres1) & (s_g < smax*thres2)
    if keep.sum() == 0:
        raise ValueError(f"expert : bande vide ({s_g.min():.2e}, {s_g.max():.2e})")
    T = V_g[:, keep]/np.sqrt(s_g[keep]); r = int(keep.sum())

    n_tr = len(X_train)
    A_qp = np.hstack([(Phi_train - mean_phi)@T, np.ones((n_tr, 1))])
    G_qp = np.eye(r+1); G_qp[-1, -1] = const_pen
    sol, feas, marge = solve_qp(A_qp, y_train, G_qp, qp_margin)
    coef = T@sol[:r]; offset = sol[-1] - mean_phi@coef
    return {'P': P, 'poly': poly, 'powers': powers, 'lo': lo, 'span': span, 'SC': SC,
            'coef': coef, 'offset': offset, 'r': r, 'nf': n_feat,
            'in_band': int(keep.sum()), 'feas': feas, 'marge': marge,
            'norm_H': float(np.sqrt(max(sol[:r]@sol[:r], 0)))}

def expert_predict(expert, X):
    Z = X @ expert['P'].T
    U = 2*(Z - expert['lo'])/expert['span'] - 1.0
    return expert['poly'].transform(U) @ expert['coef'] + expert['offset']

def expert_features_and_grads_on_X(expert, X):
    Z = X @ expert['P'].T
    U = 2*(Z - expert['lo'])/expert['span'] - 1.0
    powers = expert['powers']; coef = expert['coef']; SC = expert['SC']
    P = expert['P']; k = P.shape[0]
    Phi = expert['poly'].transform(U)
    f = Phi @ coef + expert['offset']

    def pderiv(Uu, dims):
        Pw = powers.astype(float).copy(); m = coef.astype(float).copy()
        for dm in dims: m = m*Pw[:, dm]; Pw[:, dm] -= 1
        v = m != 0
        return np.zeros(len(Uu)) if not v.any() else (Uu[:, None, :]**Pw[None, v, :]).prod(2)@m[v]

    grad_z = np.zeros((len(U), k))
    for a in range(k):
        grad_z[:, a] = SC[a] * pderiv(U, (a,))       # ∂f/∂z_a
    grad_x = grad_z @ P                               # (n, d)
    return f, grad_x


# ── Phase 1 : construire n_experts experts ───────────────────────────────────
rng = np.random.default_rng(params["seed"])
n_amb = X_train.shape[1]
experts = []
for i in range(params["n_experts"]):
    P = random_projection(n_amb, params["k_proj"], rng)
    e = fit_expert(X_all, X_train, y_train, P,
                   params["n_deg"], params["weights_plg"], params["lambda_G"],
                   params["qp_margin"], params["thres1"], params["thres2"],
                   params["const_pen"],
                   params["emb_thres1_expert"], params["embed_dim_expert"])
    experts.append(e)
    f_te = expert_predict(e, X_test)
    def acc(f, y):
        sg = np.sign(f)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
    from sklearn.metrics import roc_auc_score
    try: auc = roc_auc_score((y_test > 0).astype(int), f_te)
    except Exception: auc = float('nan')
    print(f"  expert {i+1}/{params['n_experts']} | rang={e['r']}/{e['in_band']}/{e['nf']} | "
          f"faisable={e['feas']} marge={e['marge']:.3f} ‖g‖_H={e['norm_H']:.4f} | "
          f"acc_te={acc(f_te, y_test):.4f} AUC={auc:.4f}")

# ── Sélecteur d'experts par ‖g‖_H (avant Phase 2) ────────────────────────────
# Les experts avec ‖g‖_H très grande sont oscillants et parasitent la norme H
# de la combinaison Phase 2. On les jette avant de construire G_H.
norms_H = np.array([e['norm_H'] for e in experts])
min_norm = float(np.min(norms_H))
K_sel = params["K_norm_selector"]
kept_mask = norms_H <= K_sel * min_norm
n_kept = int(kept_mask.sum())
print(f"\nSélecteur ‖g‖_H : median={min_norm:.3f}, seuil={K_sel}·median={K_sel*min_norm:.3f}")
print(f"  gardés : {n_kept}/{len(experts)}   jetés : {len(experts)-n_kept} experts")
if n_kept < len(experts):
    jetted = [i for i in range(len(experts)) if not kept_mask[i]]
    print(f"  indices jetés : {jetted}")
experts = [e for i, e in enumerate(experts) if kept_mask[i]]

# ── Phase 2 : QP sur les experts avec Gram H(cloud) ──────────────────────────
print("\nPhase 2 : QP sur les experts avec Gram H(cloud)...")
n_exp = len(experts)
# calcul de G_H[i,j] = <f_i, f_j>_H(cloud) par Monte Carlo sur X_all
F_all  = np.zeros((len(X_all), n_exp))
G_all  = np.zeros((len(X_all), n_exp, n_amb))     # gradients ∇_x f_i sur X_all
F_train_e = np.zeros((len(X_train), n_exp))
F_test_e  = np.zeros((len(X_test),  n_exp))
for i, e in enumerate(experts):
    fi_all, gi_all = expert_features_and_grads_on_X(e, X_all)
    F_all[:, i] = fi_all; G_all[:, i, :] = gi_all
    F_train_e[:, i] = expert_predict(e, X_train)
    F_test_e[:, i]  = expert_predict(e, X_test)

n_cloud = len(X_all)
w0 = params["weights_plg"].get(0, 0.); w1 = params["weights_plg"].get(1, 0.)
G_H = np.zeros((n_exp, n_exp))
if w0: G_H += w0 * (F_all.T @ F_all) / n_cloud
if w1: G_H += w1 * np.einsum('nid,njd->ij', G_all, G_all) / n_cloud
G_H += 1e-10 * np.eye(n_exp)                # petite régularisation numérique

# QP Phase 2 : min α^T G_H α  s.c.  y_j · Σ_i α_i f_i(x_j) >= marge
n_tr = len(X_train)
A2 = np.hstack([F_train_e, np.ones((n_tr, 1))])   # + biais libre
G2 = np.zeros((n_exp+1, n_exp+1)); G2[:n_exp, :n_exp] = G_H
G2[-1, -1] = params["const_pen"]
alpha_full, feas2, marge2 = solve_qp(A2, y_train, G2, params["qp_margin"])
alpha = alpha_full[:n_exp]; off_phase2 = alpha_full[-1]

pred_train = F_train_e @ alpha + off_phase2
pred_test  = F_test_e  @ alpha + off_phase2

def _acc(f, y):
    sg = np.sign(f)
    return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
from sklearn.metrics import roc_auc_score
try: auc_p2 = roc_auc_score((y_test > 0).astype(int), pred_test)
except Exception: auc_p2 = float('nan')
print(f"  QP Phase 2 : faisable={feas2} marge={marge2:.3f}")
print(f"  alpha = {alpha.round(3)}")
print(f"  train : acc={_acc(pred_train, y_train):.4f}   "
      f"test : acc={_acc(pred_test, y_test):.4f} AUC={auc_p2:.4f}")

print("\n" + "="*72)
print("BILAN")
print("="*72)
print(f"  QP ambiant (deg {params['deg_clf_avant']}) : acc={acc0:.4f} AUC={auc0:.4f}")
print(f"  Experts + Phase 2 QP        : acc={_acc(pred_test, y_test):.4f} AUC={auc_p2:.4f}")


Fashion-MNIST {manteau} vs {pullover}, 7×7 -> dim 49
  train    : 10 pts  (5 × {manteau}, 5 × {pullover})
  test     : 200 pts  (100 × {manteau}, 100 × {pullover})
  unlabeled: 2000 pts  (labels cachés pour le fit)

AVANT PLONGEMENT (espace ambiant)
  [séparation ambiant] inter/intra = 1.0540   (inter=9.286 intra=8.811)   1-NN croisé = 0.185
